In [1]:
import requests
import pandas as pd
from pathlib import Path


OUT_DIR = Path("api_column_checks")
OUT_DIR.mkdir(exist_ok=True)


def show_columns(name, url, params=None):
    print(f"\n--- {name} ---")
    print("URL:", url)
    print("PARAMS:", params)

    r = requests.get(url, params=params, timeout=60)
    print("STATUS:", r.status_code)
    r.raise_for_status()

    payload = r.json()

    if isinstance(payload, dict) and "data" in payload:
        records = payload["data"]
    elif isinstance(payload, dict) and "result" in payload and "records" in payload["result"]:
        records = payload["result"]["records"]
    elif isinstance(payload, list):
        records = payload
    else:
        print("Unknown response shape:")
        print(payload.keys() if isinstance(payload, dict) else type(payload))
        return

    df = pd.DataFrame(records)

    print("ROWS:", len(df))
    print("COLUMNS:")
    for c in df.columns:
        print("  -", c)

    df.head(5).to_csv(OUT_DIR / f"{name}_sample.csv", index=False)


# ============================================================
# 1. NESO Historic Demand Data
# ============================================================

# Example: 2023 Historic Demand Data resource
# NESO uses CKAN datastore_search for its data portal resources.
# Source docs: NESO Historic Demand Data + NESO API guidance.
show_columns(
    name="neso_historic_demand_2023",
    url="https://api.neso.energy/api/3/action/datastore_search",
    params={
        "resource_id": "bf5ab335-9b40-4ea4-b93a-ab4af7bce003",
        "limit": 5,
    }
)


# ============================================================
# 2. Elexon BM Units reference
# ============================================================

# This should show fields like:
# nationalGridBmUnit, elexonBmUnit, fuelType, gspGroupId, gspGroupName, etc.
show_columns(
    name="elexon_bmunits_reference",
    url="https://data.elexon.co.uk/bmrs/api/v1/reference/bmunits/all",
    params={
        "format": "json"
    }
)


# ============================================================
# 3. Elexon B1610 generation
# ============================================================

# B1610 stream: actual metered volume output per BMU per settlement period.
# Small one-day request only, just to inspect columns.
show_columns(
    name="elexon_b1610_sample",
    url="https://data.elexon.co.uk/bmrs/api/v1/datasets/B1610/stream",
    params={
        "from": "2023-01-01T00:00Z",
        "to": "2023-01-02T00:00Z",
        "settlementPeriodFrom": 1,
        "settlementPeriodTo": 48,
        "format": "json",
    }
)


--- neso_historic_demand_2023 ---
URL: https://api.neso.energy/api/3/action/datastore_search
PARAMS: {'resource_id': 'bf5ab335-9b40-4ea4-b93a-ab4af7bce003', 'limit': 5}
STATUS: 200
ROWS: 5
COLUMNS:
  - _id
  - SETTLEMENT_DATE
  - SETTLEMENT_PERIOD
  - ND
  - TSD
  - ENGLAND_WALES_DEMAND
  - EMBEDDED_WIND_GENERATION
  - EMBEDDED_WIND_CAPACITY
  - EMBEDDED_SOLAR_GENERATION
  - EMBEDDED_SOLAR_CAPACITY
  - NON_BM_STOR
  - PUMP_STORAGE_PUMPING
  - SCOTTISH_TRANSFER
  - IFA_FLOW
  - IFA2_FLOW
  - BRITNED_FLOW
  - MOYLE_FLOW
  - EAST_WEST_FLOW
  - NEMO_FLOW
  - NSL_FLOW
  - ELECLINK_FLOW
  - VIKING_FLOW
  - GREENLINK_FLOW

--- elexon_bmunits_reference ---
URL: https://data.elexon.co.uk/bmrs/api/v1/reference/bmunits/all
PARAMS: {'format': 'json'}
STATUS: 200
ROWS: 3004
COLUMNS:
  - nationalGridBmUnit
  - elexonBmUnit
  - eic
  - fuelType
  - leadPartyName
  - bmUnitType
  - fpnFlag
  - bmUnitName
  - leadPartyId
  - demandCapacity
  - generationCapacity
  - productionOrConsumptionFlag
  - tra